# Retraining после деградации

Есть гипотеза, что при обычном переобучении большая старая история продолжает задавать модели прежние закономерности. Проверим это на одной и той же CatBoost, меняя только выбор строк и их веса.

Силу recency weighting и размер окна выбираем по апрелю. Май–август смотрим только после выбора как финальный monitoring-период. Более поздних данных в датасете нет, поэтому это ретроспективная проверка, а не новая независимая оценка после реального деплоя.

## 1. Временные периоды

In [1]:
import json
from pathlib import Path

from catboost import CatBoostClassifier
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

train = pd.read_parquet(ROOT / "data/processed/train.parquet")
validation = pd.read_parquet(ROOT / "data/processed/validation.parquet")
first_batch = pd.read_parquet(ROOT / "data/processed/test.parquet")
production = pd.read_parquet(ROOT / "data/processed/production.parquet")

history = pd.concat([train, validation, first_batch], ignore_index=True)
history = history.sort_values("booking_date")

adaptation_train = history[history["booking_date"] < "2017-04-01"].copy()
april_validation = history[
    history["booking_date"].between("2017-04-01", "2017-04-30")
].copy()

In [2]:
pd.DataFrame({
    "rows": [len(train), len(validation), len(adaptation_train), len(april_validation), len(production)],
    "date_from": [
        train["booking_date"].min(),
        validation["booking_date"].min(),
        adaptation_train["booking_date"].min(),
        april_validation["booking_date"].min(),
        production["booking_date"].min(),
    ],
    "date_to": [
        train["booking_date"].max(),
        validation["booking_date"].max(),
        adaptation_train["booking_date"].max(),
        april_validation["booking_date"].max(),
        production["booking_date"].max(),
    ],
}, index=["old train", "old validation", "adaptation train", "april validation", "may–august"])

,rows,date_from,date_to
old train,82629,2013-06-24,2016-10-31
old validation,18065,2016-11-01,2017-01-31
adaptation train,110075,2013-06-24,2017-03-31
april validation,2736,2017-04-01,2017-04-30
may–august,6579,2017-05-01,2017-08-31


К этому моменту ответы за февраль и март уже известны, поэтому их можно добавить в обучение. Апрель нужен для выбора стратегии и порога, а май–август в настройке не участвуют.

## 2. Одинаковая модель для всех стратегий

In [3]:
target = "is_canceled"
drop_columns = [target, "arrival_date", "booking_date"]
categorical_columns = adaptation_train.drop(columns=drop_columns).select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

catboost_params = {
    "iterations": 436,
    "learning_rate": 0.04,
    "depth": 8,
    "l2_leaf_reg": 1,
    "random_strength": 0,
    "bagging_temperature": 1,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "random_seed": 42,
    "verbose": False,
    "allow_writing_files": False,
}


def prepare_catboost(data):
    X = data.drop(columns=drop_columns).copy()
    for column in categorical_columns:
        X[column] = X[column].astype("string").fillna("Missing")
    return X, data[target]


def train_catboost(data, sample_weight=None):
    X, y = prepare_catboost(data)
    model = CatBoostClassifier(**catboost_params)
    model.fit(
        X,
        y,
        cat_features=categorical_columns,
        sample_weight=sample_weight,
    )
    return model


def model_metrics(model, data, threshold=0.5):
    X, y = prepare_catboost(data)
    probabilities = model.predict_proba(X)[:, 1]
    predictions = (probabilities >= threshold).astype(int)
    return {
        "precision": precision_score(y, predictions, zero_division=0),
        "recall": recall_score(y, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y, probabilities),
        "pr_auc": average_precision_score(y, probabilities),
    }


def choose_threshold(model, data):
    X, y = prepare_catboost(data)
    probabilities = model.predict_proba(X)[:, 1]
    precision, recall, thresholds = precision_recall_curve(y, probabilities)
    table = pd.DataFrame({
        "threshold": thresholds,
        "precision": precision[:-1],
        "recall": recall[:-1],
    })
    best = (
        table[table["precision"] >= 0.70]
        .sort_values(["recall", "precision"], ascending=False)
        .iloc[0]
    )
    return float(best["threshold"])

Чтобы сравнение было честным, параметры, признаки и обработка категорий везде одинаковые. Отличаются только обучающие строки и их веса.

## 3. Full history, recency weighting и recent window

In [4]:
def recency_weights(dates, half_life_months):
    age_in_months = (dates.max() - dates).dt.days / 30.4
    weights = 0.5 ** (age_in_months / half_life_months)
    return weights / weights.mean()

Смысл `half_life_months` простой: через указанное число месяцев вес наблюдения уменьшается вдвое. Средний вес приводится к 1, чтобы менять влияние старых и свежих строк, а не общий масштаб весов.

In [5]:
original_model = train_catboost(train)
full_history_model = train_catboost(adaptation_train)

weighting_models = {}
for half_life in [6, 12, 18]:
    weights = recency_weights(adaptation_train["booking_date"], half_life)
    weighting_models[half_life] = train_catboost(adaptation_train, weights)

window_models = {}
for months in [6, 12]:
    cutoff = adaptation_train["booking_date"].max() - pd.DateOffset(months=months)
    window_data = adaptation_train[adaptation_train["booking_date"] >= cutoff]
    window_models[months] = train_catboost(window_data)

## 4. Выбор силы weighting и размера окна по апрелю

In [6]:
option_results = []

for half_life, model in weighting_models.items():
    option_results.append({
        "option": f"Weighting, half-life {half_life} months",
        **model_metrics(model, april_validation),
    })

for months, model in window_models.items():
    option_results.append({
        "option": f"Recent window, {months} months",
        **model_metrics(model, april_validation),
    })

option_table = pd.DataFrame(option_results).set_index("option")
option_table[["roc_auc", "pr_auc"]].sort_values("pr_auc", ascending=False).round(3)

,roc_auc,pr_auc
option,,
"Weighting, half-life 18 months",0.873,0.748
"Weighting, half-life 6 months",0.872,0.745
"Weighting, half-life 12 months",0.873,0.745
"Recent window, 12 months",0.875,0.740
"Recent window, 6 months",0.868,0.735


In [7]:
best_half_life = int(
    option_table.loc[option_table.index.str.startswith("Weighting"), "pr_auc"]
    .idxmax()
    .split()[-2]
)
best_window = int(
    option_table.loc[option_table.index.str.startswith("Recent window"), "pr_auc"]
    .idxmax()
    .split()[-2]
)

strategy_models = {
    "Original / old data": original_model,
    "Full history": full_history_model,
    "Recency weighting": weighting_models[best_half_life],
    "Recent window": window_models[best_window],
}

print("Выбранный half-life:", best_half_life, "месяцев")
print("Выбранное окно:", best_window, "месяцев")

Выбранный half-life: 18 месяцев
Выбранное окно: 12 месяцев


## 5. Два временных backtest-периода

In [8]:
old_weighted_model = train_catboost(
    train,
    recency_weights(train["booking_date"], best_half_life),
)
old_window_cutoff = train["booking_date"].max() - pd.DateOffset(months=best_window)
old_window_model = train_catboost(train[train["booking_date"] >= old_window_cutoff])

old_fold_models = {
    "Original / old data": original_model,
    "Full history": original_model,
    "Recency weighting": old_weighted_model,
    "Recent window": old_window_model,
}

comparison_rows = []
for strategy in strategy_models:
    comparison_rows.append({
        "strategy": strategy,
        "old_validation_pr_auc": model_metrics(
            old_fold_models[strategy], validation
        )["pr_auc"],
        "april_pr_auc": model_metrics(
            strategy_models[strategy], april_validation
        )["pr_auc"],
    })

validation_comparison = pd.DataFrame(comparison_rows).set_index("strategy")
validation_comparison.round(3)

,old_validation_pr_auc,april_pr_auc
strategy,,
Original / old data,0.897,0.726
Full history,0.897,0.743
Recency weighting,0.898,0.748
Recent window,0.896,0.740


Для старой validation стратегии обучаются только на данных до ноября. Иначе часть моделей уже видела бы оцениваемые месяцы, и сравнение получилось бы нечестным.

На апреле recency weighting немного лучше full history, но разница небольшая. Recent window работает слабее и сильнее теряет качество на старом периоде.

## 6. Финальный monitoring-период

In [9]:
thresholds = {
    "Original / old data": choose_threshold(original_model, validation),
    "Full history": choose_threshold(full_history_model, april_validation),
    "Recency weighting": choose_threshold(
        weighting_models[best_half_life], april_validation
    ),
    "Recent window": choose_threshold(window_models[best_window], april_validation),
}

final_rows = []
for strategy, model in strategy_models.items():
    final_rows.append({
        "strategy": strategy,
        "threshold": thresholds[strategy],
        **model_metrics(model, production, thresholds[strategy]),
    })

final_comparison = pd.DataFrame(final_rows).set_index("strategy")
final_comparison.round(3)

,threshold,precision,recall,roc_auc,pr_auc
strategy,,,,,
Original / old data,0.398,0.550,0.746,0.852,0.654
Full history,0.389,0.661,0.540,0.859,0.685
Recency weighting,0.361,0.647,0.558,0.858,0.680
Recent window,0.358,0.674,0.561,0.860,0.672


Небольшой выигрыш weighting на апреле не повторился на май–август. Full history показал такой же общий уровень и немного лучший PR-AUC, а recent window тоже не выиграл.

В этой ситуации обычное переобучение на всей истории выглядит разумнее: оно проще, а подтверждения, что старые данные обязательно нужно ослаблять, не получилось.

## 7. Current model и candidate model

In [10]:
current_preprocessor = joblib.load(ROOT / "models/preprocessor.joblib")
current_model = joblib.load(ROOT / "models/lightgbm_v1.joblib")

with (ROOT / "models/lightgbm_v1_metadata.json").open(encoding="utf-8") as file:
    current_metadata = json.load(file)

X_current = production.drop(columns=current_metadata["drop_cols"]).copy()
for column in current_metadata["categorical_cols"]:
    X_current[column] = X_current[column].astype("object").where(
        X_current[column].notna(), np.nan
    )

current_proba = current_model.predict_proba(
    current_preprocessor.transform(X_current)
)[:, 1]
current_pred = (current_proba >= current_metadata["threshold"]).astype(int)

current_metrics = {
    "precision": precision_score(production[target], current_pred),
    "recall": recall_score(production[target], current_pred),
    "roc_auc": roc_auc_score(production[target], current_proba),
    "pr_auc": average_precision_score(production[target], current_proba),
}

candidate_strategy = "Full history"
candidate_model = strategy_models[candidate_strategy]
candidate_threshold = thresholds[candidate_strategy]
candidate_metrics = model_metrics(candidate_model, production, candidate_threshold)

deployment_comparison = pd.DataFrame(
    [current_metrics, candidate_metrics],
    index=["current: LightGBM v1", "candidate: CatBoost v2"],
)
deployment_comparison.round(3)

,precision,recall,roc_auc,pr_auc
current: LightGBM v1,0.575,0.765,0.857,0.661
candidate: CatBoost v2,0.661,0.540,0.859,0.685


In [11]:
deploy_candidate = (
    candidate_metrics["pr_auc"] > current_metrics["pr_auc"]
    and candidate_metrics["precision"] > current_metrics["precision"]
)

deploy_candidate

True

CatBoost лучше текущей LightGBM по PR-AUC и даёт более точный список срабатываний. Значит, её можно сохранить как новую версию.

При этом Precision на monitoring-периоде всё равно ниже цели 0.70. Эту проблему новая модель полностью не решила, поэтому после выкладки качество нужно продолжать отслеживать.

## 8. Сохранение CatBoost v2

In [12]:
if deploy_candidate:
    model_dir = ROOT / "models"
    candidate_model.save_model(model_dir / "catboost_v2.cbm")

    X_candidate, _ = prepare_catboost(adaptation_train)
    model_metadata = {
        "model": "CatBoost",
        "version": "v2",
        "training_strategy": "full_history",
        "threshold": candidate_threshold,
        "feature_cols": X_candidate.columns.tolist(),
        "categorical_cols": categorical_columns,
        "trained_until": "2017-03-31",
        "validation_period": "2017-04",
        "monitoring_period": "2017-05 to 2017-08",
    }

    with (model_dir / "model_metadata.json").open("w", encoding="utf-8") as file:
        json.dump(model_metadata, file, indent=2, ensure_ascii=False)

## Итог

Recency weighting немного помог на апреле, но не удержал преимущество на следующем периоде. Recent window тоже не оказался лучше. Поэтому для CatBoost v2 остаётся самый простой вариант — обучение на всей доступной истории.

Дальше логика такая: новый batch показывает drift и падение качества → обучаем candidate → сравниваем его с текущей моделью → заменяем модель только при реальном улучшении. Сам по себе drift ничего автоматически не запускает.